# Colab Inference: Qwen 3B LoRA v2

Use this notebook on Google Colab to load the saved `qwen3b_svg_lora_v2` adapter and generate a submission without retraining.

Before running:
- Switch Colab runtime to **GPU**.
- Copy `artifacts/qwen3b_svg_lora_v2` into your Google Drive project folder.
- Make sure `data/test.csv` is available in that same project folder structure.

In [ ]:
# Install Colab dependencies.
%pip install -q unsloth transformers accelerate peft bitsandbytes pandas lxml sentencepiece tiktoken

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# Change this if your Drive folder is somewhere else.
PROJECT_DIR = Path('/content/drive/MyDrive/DL_M1')
ADAPTER_DIR = PROJECT_DIR / 'artifacts/qwen3b_svg_lora_v2'
TEST_CSV_PATH = PROJECT_DIR / 'data/test.csv'
SUBMISSION_PATH = PROJECT_DIR / 'submission_qwen3b_v2.csv'

if not ADAPTER_DIR.exists():
    raise FileNotFoundError(f'Adapter directory not found: {ADAPTER_DIR}')
if not (ADAPTER_DIR / 'adapter_model.safetensors').exists():
    raise FileNotFoundError(f'Missing adapter weights in: {ADAPTER_DIR}')
if not TEST_CSV_PATH.exists():
    raise FileNotFoundError(f'Test CSV not found: {TEST_CSV_PATH}')

print(f'Adapter dir: {ADAPTER_DIR}')
print(f'Test CSV: {TEST_CSV_PATH}')
print(f'Submission path: {SUBMISSION_PATH}')

In [ ]:
import json
import random
import re
import shutil
import time
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError('This notebook expects a Colab GPU runtime.')

adapter_config = json.loads((ADAPTER_DIR / 'adapter_config.json').read_text())
BASE_MODEL_NAME = adapter_config['base_model_name_or_path']
HF_BASE_MODEL_NAME = (
    'Qwen/Qwen2.5-3B-Instruct'
    if BASE_MODEL_NAME.startswith('unsloth/')
    else BASE_MODEL_NAME
)

FAST_MODE = True
MAX_SEQ_LENGTH = 1024 if FAST_MODE else 1536
MAX_NEW_TOKENS = 1024 if FAST_MODE else 2000
BATCH_SIZE = 8
DO_SAMPLE = not FAST_MODE
TEMPERATURE = 0.2
TOP_P = 0.9
REPETITION_PENALTY = 1.15 if FAST_MODE else 1.3
LOCAL_TMP_DIR = Path('/content/qwen3b_v2_fast_tmp') if FAST_MODE else PROJECT_DIR
LOCAL_TMP_DIR.mkdir(parents=True, exist_ok=True)

print(f'Torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Adapter base model: {BASE_MODEL_NAME}')
print(f'Loading base model as: {HF_BASE_MODEL_NAME}')
print(f'FAST_MODE: {FAST_MODE}')
print(f'Max seq length: {MAX_SEQ_LENGTH}')
print(f'Max new tokens: {MAX_NEW_TOKENS}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Do sample: {DO_SAMPLE}')
print(f'Temp output dir: {LOCAL_TMP_DIR}')

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
)

base_model = AutoModelForCausalLM.from_pretrained(
    HF_BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
model.eval()
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = True

MODEL_DEVICE = next(model.parameters()).device
print(f'Loaded model on: {MODEL_DEVICE}')

In [ ]:
SYSTEM_PROMPT = (
    'You generate compact, valid SVG markup from user requests. '
    'Return only SVG code with a single root <svg> element.'
)

SVG_REGEX = re.compile(r'<svg[\s\S]*?</svg>', flags=re.IGNORECASE)
SVG_OPEN_RE = re.compile(r'<svg[\s\S]*', flags=re.IGNORECASE)
_BARE_AMP_RE = re.compile(r'&(?![a-zA-Z][a-zA-Z0-9]*;|#[0-9]+;|#x[0-9a-fA-F]+;)')


def repair_svg(svg_text):
    return _BARE_AMP_RE.sub('&amp;', svg_text)


def extract_svg(text):
    match = SVG_REGEX.search(text)
    if match:
        return repair_svg(match.group(0).strip())

    match = SVG_OPEN_RE.search(text)
    if match:
        truncated = repair_svg(match.group(0).strip()) + '</svg>'
        try:
            root = ET.fromstring(truncated)
            if root.tag.endswith('svg'):
                return truncated
        except ET.ParseError:
            pass
    return ''


def is_valid_svg(svg_text):
    if not svg_text:
        return False
    try:
        root = ET.fromstring(svg_text)
        return root.tag.endswith('svg')
    except ET.ParseError:
        return False


def fallback_svg(_prompt):
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect x="0" y="0" width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )


def build_chat_text(prompt):
    return (
        '<|im_start|>system\n'
        f'{SYSTEM_PROMPT}<|im_end|>\n'
        '<|im_start|>user\n'
        f'{prompt}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )


def build_generate_kwargs(max_new_tokens):
    kwargs = {
        'max_new_tokens': max_new_tokens,
        'repetition_penalty': REPETITION_PENALTY,
        'use_cache': True,
        'pad_token_id': tokenizer.eos_token_id,
        'eos_token_id': tokenizer.eos_token_id,
    }
    if DO_SAMPLE:
        kwargs.update({
            'do_sample': True,
            'temperature': TEMPERATURE,
            'top_p': TOP_P,
        })
    else:
        kwargs['do_sample'] = False
    return kwargs


def generate_svg(prompt, max_new_tokens=MAX_NEW_TOKENS, debug=False):
    inputs = tokenizer(
        build_chat_text(prompt),
        return_tensors='pt',
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(MODEL_DEVICE)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            **build_generate_kwargs(max_new_tokens),
        )

    decoded = tokenizer.decode(output_ids[0, input_len:], skip_special_tokens=True)
    if debug:
        print('=== RAW MODEL OUTPUT ===')
        print(repr(decoded[:500]))
        print('========================')
    svg = extract_svg(decoded)
    if not is_valid_svg(svg):
        svg = fallback_svg(prompt)
    return svg


def generate_svg_batch(prompts, max_new_tokens=MAX_NEW_TOKENS):
    chat_texts = [build_chat_text(prompt) for prompt in prompts]
    inputs = tokenizer(
        chat_texts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    ).to(MODEL_DEVICE)
    input_len = inputs['input_ids'].shape[1]

    try:
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                **build_generate_kwargs(max_new_tokens),
            )
        decoded_batch = tokenizer.batch_decode(output_ids[:, input_len:], skip_special_tokens=True)
    except RuntimeError as exc:
        if 'broadcast shape' not in str(exc):
            raise
        print('Batch generation hit a shape mismatch; retrying prompts one by one.', flush=True)
        torch.cuda.empty_cache()
        decoded_batch = []
        for idx, prompt in enumerate(prompts, start=1):
            print(f'  single-prompt retry {idx}/{len(prompts)}', flush=True)
            inputs = tokenizer(
                build_chat_text(prompt),
                return_tensors='pt',
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
            ).to(MODEL_DEVICE)
            single_input_len = inputs['input_ids'].shape[1]
            with torch.no_grad():
                output_ids = model.generate(
                    **inputs,
                    **build_generate_kwargs(max_new_tokens),
                )
            decoded_batch.append(tokenizer.decode(output_ids[0, single_input_len:], skip_special_tokens=True))

    svgs = []
    failures = []
    for prompt, decoded in zip(prompts, decoded_batch):
        svg = extract_svg(decoded)
        parse_error = None
        valid = False
        if svg:
            try:
                root = ET.fromstring(svg)
                valid = root.tag.endswith('svg')
            except ET.ParseError as exc:
                parse_error = str(exc)

        if not valid:
            failures.append({
                'prompt': prompt,
                'raw_decoded': decoded[:2000],
                'extracted_svg': svg[:1000] if svg else '',
                'no_svg_found': not bool(svg),
                'parse_error': parse_error,
            })
            svg = fallback_svg(prompt)
        svgs.append(svg)

    return svgs, failures

In [ ]:
test_prompt = 'a simple blue bird icon'
pred_svg = generate_svg(test_prompt, max_new_tokens=MAX_NEW_TOKENS, debug=True)
print(pred_svg[:500])
print('Used fallback:', pred_svg == fallback_svg(test_prompt))
print('Valid SVG:', is_valid_svg(pred_svg))

In [ ]:
_run_ts = int(time.time())
DRIVE_SUBMISSION_PATH = PROJECT_DIR / f'submission_qwen3b_v2_{_run_ts}.csv'
DRIVE_DEBUG_LOG_PATH = PROJECT_DIR / f'debug_qwen3b_v2_{_run_ts}.jsonl'
LOCAL_SUBMISSION_PATH = LOCAL_TMP_DIR / DRIVE_SUBMISSION_PATH.name
LOCAL_DEBUG_LOG_PATH = LOCAL_TMP_DIR / DRIVE_DEBUG_LOG_PATH.name

print(f'Will write local submission to: {LOCAL_SUBMISSION_PATH}')
print(f'Will copy final submission to: {DRIVE_SUBMISSION_PATH}')
print(f'Will write local debug log to: {LOCAL_DEBUG_LOG_PATH}')
print(f'Will copy final debug log to: {DRIVE_DEBUG_LOG_PATH}')

test_df = pd.read_csv(TEST_CSV_PATH)
invalid_count = 0
t0 = time.time()

tmp_path = Path(str(LOCAL_SUBMISSION_PATH) + '.part')
done_ids = set()
if tmp_path.exists():
    print(f'Resuming inference from existing checkpoint file: {tmp_path}')
    done_ids = set(pd.read_csv(tmp_path)['id'].tolist())
elif LOCAL_SUBMISSION_PATH.exists():
    print(f'Continuing from existing local output: {LOCAL_SUBMISSION_PATH}')
    shutil.copy2(LOCAL_SUBMISSION_PATH, tmp_path)
    done_ids = set(pd.read_csv(tmp_path)['id'].tolist())
elif DRIVE_SUBMISSION_PATH.exists():
    print(f'Continuing from existing Drive output: {DRIVE_SUBMISSION_PATH}')
    shutil.copy2(DRIVE_SUBMISSION_PATH, tmp_path)
    done_ids = set(pd.read_csv(tmp_path)['id'].tolist())
    if DRIVE_DEBUG_LOG_PATH.exists() and not LOCAL_DEBUG_LOG_PATH.exists():
        shutil.copy2(DRIVE_DEBUG_LOG_PATH, LOCAL_DEBUG_LOG_PATH)

rows_buffer = []
total = len(test_df)

for start in range(0, total, BATCH_SIZE):
    batch_df = test_df.iloc[start:start + BATCH_SIZE]
    batch_df = batch_df[~batch_df['id'].isin(done_ids)]
    if batch_df.empty:
        continue

    prompts = batch_df['prompt'].tolist()
    ids = batch_df['id'].tolist()
    batch_start = start + 1
    batch_end = start + len(batch_df)
    print(f'Starting batch rows {batch_start}-{batch_end} of {total}', flush=True)

    svgs, failures = generate_svg_batch(prompts, max_new_tokens=MAX_NEW_TOKENS)

    if failures:
        with open(LOCAL_DEBUG_LOG_PATH, 'a') as dbf:
            for entry in failures:
                dbf.write(json.dumps(entry) + '\n')

    rows_buffer.clear()
    for sample_id, prompt, svg in zip(ids, prompts, svgs):
        if not is_valid_svg(svg):
            invalid_count += 1
            svg = fallback_svg(prompt)
        rows_buffer.append({'id': sample_id, 'svg': svg})

    pd.DataFrame(rows_buffer).to_csv(
        tmp_path,
        mode='a',
        index=False,
        header=not tmp_path.exists(),
    )
    done_ids.update([row['id'] for row in rows_buffer])

    elapsed = (time.time() - t0) / 60
    print(
        f'Progress: {len(done_ids)}/{total} rows written, elapsed={elapsed:.1f}m, fallbacks={invalid_count}',
        flush=True,
    )

if tmp_path.exists():
    tmp_path.replace(LOCAL_SUBMISSION_PATH)

shutil.copy2(LOCAL_SUBMISSION_PATH, DRIVE_SUBMISSION_PATH)
if LOCAL_DEBUG_LOG_PATH.exists():
    shutil.copy2(LOCAL_DEBUG_LOG_PATH, DRIVE_DEBUG_LOG_PATH)

elapsed_min = (time.time() - t0) / 60
sub_df = pd.read_csv(LOCAL_SUBMISSION_PATH)
print(f'Local submission: {LOCAL_SUBMISSION_PATH}')
print(f'Drive submission: {DRIVE_SUBMISSION_PATH}')
print(f'Local debug log: {LOCAL_DEBUG_LOG_PATH}')
print(f'Drive debug log: {DRIVE_DEBUG_LOG_PATH}')
print(f'Rows: {len(sub_df)}')
print(f'Invalid/fallback count: {invalid_count}')
print(f'Runtime (minutes): {elapsed_min:.2f}')
sub_df.head()